# AXE 2 — 02. ALS Model (Netflix)
_Notebook 2/3 — ALS sur interactions Netflix → `data/warehouse/als_scores`_

## Pipeline
1. Charger les interactions Netflix (virtual users)
2. Log-scaling des poids
3. Entraîner ALS (implicitPrefs=True)
4. Matcher les titres d'ancrage via Spark natif (contains)
5. Similarité cosinus via fonctions array Spark
6. Boost ancres + normalisation percent_rank()
7. Écrire als_scores

**Output** : `data/warehouse/als_scores`  
Colonnes : `item_id`, `item_title`, `platform`, `play_count`, `predicted_score`, `rank`

In [ ]:
import os, sys
import numpy as np
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType, FloatType, StringType
from pyspark.ml.recommendation import ALS
from pyspark.sql.window import Window

sys.path.insert(0, os.path.abspath(os.path.join(os.path.dirname("__file__"), "../../..")))
from config import WAREHOUSE, NETFLIX_ANCHOR_TITLES

spark = SparkSession.builder \
    .appName("MyDigitalTwin-ALS-Netflix") \
    .master("local[*]") \
    .config("spark.driver.memory", "4g") \
    .config("spark.sql.shuffle.partitions", "8") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

In [ ]:
# ── 1. ANCRES NETFLIX (Spark DataFrame) ───────────────────────────────────────
# NETFLIX_ANCHOR_TITLES : titres de référence définis dans config.py
# Normalisation identique à celle appliquée sur item_title dans les interactions.

anchors_df = spark.createDataFrame(
    [(t,) for t in NETFLIX_ANCHOR_TITLES], ["anchor_title"]
).withColumn(
    "anchor_norm",
    F.trim(F.regexp_replace(F.lower(F.col("anchor_title")), r'[^\w\s]', ''))
).cache()

print(f"{anchors_df.count()} ancres Netflix chargées.")

In [ ]:
# ── 2. CHARGEMENT + LOG-SCALING ───────────────────────────────────────────────
df = spark.read.parquet(os.path.join(WAREHOUSE, "interactions")) \
    .withColumn("user_id", F.col("user_id").cast(IntegerType())) \
    .withColumn("item_id",  F.col("item_id").cast(IntegerType())) \
    .withColumn("weight",   F.log1p(F.col("play_count").cast("double")))

# ── 3. ENTRAINEMENT ALS ───────────────────────────────────────────────────────
als = ALS(
    userCol="user_id", itemCol="item_id", ratingCol="weight",
    rank=20, maxIter=20, regParam=0.15,
    implicitPrefs=True, coldStartStrategy="drop", seed=42
)
model = als.fit(df)

# ── 4. MATCHING ANCRES (Spark natif — regexp + contains) ──────────────────────
# Agrégation des plays par item pour le scoring final
totals = df.groupBy("item_id", "item_title").agg(F.sum("play_count").alias("plays"))

items_norm = totals.withColumn(
    "title_norm",
    F.trim(F.regexp_replace(F.lower(F.col("item_title")), r'[^\w\s]', ''))
)

# broadcast : anchors_df est petit (~100 lignes)
anchor_items = items_norm.join(
    F.broadcast(anchors_df),
    items_norm.title_norm.contains(anchors_df.anchor_norm) |
    anchors_df.anchor_norm.contains(items_norm.title_norm),
    "inner"
).select("item_id", "item_title").distinct()

print(f"Profil basé sur {anchor_items.count()} ancres détectées.")

# ── 5. VECTEUR DE RÉFÉRENCE ────────────────────────────────────────────────────
# Collecte des vecteurs d'ancrage sur le driver (données petites — N_ancres lignes)
anchor_vecs = model.itemFactors \
    .join(anchor_items.select(F.col("item_id").alias("id")), on="id", how="inner") \
    .select("features") \
    .rdd.map(lambda r: r.features).collect()

ref_vec  = np.mean(anchor_vecs, axis=0)
ref_norm = (ref_vec / (np.linalg.norm(ref_vec) + 1e-8)).tolist()

# ── 6. SIMILARITÉ COSINUS (Spark array higher-order functions) ─────────────────
# zip_with  : produit élément par élément → vecteur de produits
# aggregate : somme des produits → dot product
# Pas de UDF : tout en Spark natif.

ref_col = F.array([F.lit(float(v)) for v in ref_norm])

factors_scored = model.itemFactors \
    .withColumn(
        "dot",
        F.aggregate(
            F.zip_with(F.col("features"), ref_col, lambda x, y: x * y),
            F.lit(0.0),
            lambda acc, x: acc + x
        )
    ).withColumn(
        "feat_norm",
        F.sqrt(F.aggregate(
            F.transform(F.col("features"), lambda x: x * x),
            F.lit(0.0),
            lambda acc, x: acc + x
        ))
    ).withColumn(
        "cosine_sim",
        F.col("dot") / (F.col("feat_norm") + F.lit(1e-8))
    )

# ── 7. BOOST ANCRES + NORMALISATION percent_rank() ────────────────────────────
# percent_rank() : évite un .count() séparé (job Spark supplémentaire)
# Boost 1.8 sur les ancres (titres favoris), 1.3 sinon

anchor_flag = anchor_items.select(
    F.col("item_id").alias("id"),
    F.lit(True).alias("is_anchor")
)

w_asc  = Window.orderBy(F.asc("score_boosted"))
w_rank = Window.orderBy(F.desc("score_boosted"))

result = factors_scored \
    .join(totals.select(F.col("item_id").alias("id"), "item_title", "plays"), on="id", how="inner") \
    .join(anchor_flag, on="id", how="left") \
    .withColumn("boost",
        F.when(F.col("is_anchor") == True, F.lit(1.8)).otherwise(F.lit(1.3))
    ) \
    .withColumn("score_boosted", F.col("cosine_sim") * F.col("boost")) \
    .withColumn("score", F.round(F.percent_rank().over(w_asc) * 100, 1)) \
    .withColumn("rank",  F.row_number().over(w_rank)) \
    .select("id", "item_title", "plays", "score", "rank")

In [ ]:
result.orderBy("rank").show(15, truncate=50)

In [ ]:
# ── 8. ÉCRITURE warehouse/als_scores ─────────────────────────────────────────

out = result.select(
    F.col("id").cast(IntegerType()).alias("item_id"),
    F.col("item_title").cast(StringType()),
    F.lit("netflix").cast(StringType()).alias("platform"),
    F.col("plays").cast(IntegerType()).alias("play_count"),
    F.col("score").cast(FloatType()).alias("predicted_score"),
    F.col("rank").cast(IntegerType()),
)

out.write.mode("overwrite").parquet(os.path.join(WAREHOUSE, "als_scores"))
print("Scores sauvegardés.")
spark.stop()